# 序列資料取樣

當序列太長（例如數萬個字的小說或長達數小時的感測器數據）時，我們不能直接把整條序列丟進 RNN，這會導致梯度消失或記憶體崩潰。這時，「取樣」就成了關鍵技術。

## 時間序列資料生成器 (Data Generator)

功能說明：這是一個用來「切分時間序列資料」的產生器 (generator)。會把原始資料依照長度 T 連續切成一段一段，並產生對應的 (X, Y) 訓練資料。

* X = 當前時間區段資料
* Y = 往後平移一格的資料（常見於時間序列預測，例如 RNN/LSTM）

參數說明：
* data        : 原始序列資料（例如 list 或 numpy array）
* T           : 每一段切分的長度
* start_range : 若 >0，起始點會在 0 ~ start_range 之間隨機選
* repeat      : 是否重複產生資料（True = 無限循環）

回傳：使用 yield 回傳 (X, Y, 是否為第一筆)

In [1]:
import numpy as np

def seg_data_iter_consecutive_one(data, T, start_range=0, repeat=False): 

    n = len(data)  # 取得資料總長度

    # 若有設定 start_range，則隨機選一個起始點
    if start_range > 0:
        start = np.random.randint(0, start_range)
    else:
        start = 0    

    end = n - T  # 計算最後可切分的位置

    while True:       
        # 依照長度 T 逐段往後切
        for p in range(start, end, T):

            # 取出長度為 T 的輸入資料
            X = data[p:p+T]

            # 目標資料為往後平移一格（時間序列常見做法）
            Y = data[p+1:p+T+1]  

            # 以下為曾使用但目前註解掉的資料維度處理
            #inputs = np.expand_dims(inputs, axis=1)
            #targets  = targets.reshape(-1,1)

            # 若為第一段資料，第三個參數回傳 True
            if p == start:
                yield X, Y, True
            else: 
                yield X, Y, False

        # 若重複產生資料，則重置起始點並繼續迴圈
        if not repeat:
            return

範例：建立一組 0~19 的整數列表，並透過 seg_data_iter_consecutive_one 函式將資料依指定規則分段。這裡傳入參數 (3, 5)，代表分段長度與步進長度（實際邏輯依函式定義而定），接著透過迴圈逐筆取出分段後的資料，並印出每組的 X 與 Y。

In [2]:
# 建立 0~19 的整數列表
data = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 
        10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

# 呼叫資料分段函式
data_it = seg_data_iter_consecutive_one(data, 3, 5)

# 逐筆走訪分段後的資料
for X, Y, _ in data_it:
    # 印出每組分段資料中的 X 與 Y
    # 第三個回傳值以 _ 表示不使用
    print(X, Y)

[0, 1, 2] [1, 2, 3]
[3, 4, 5] [4, 5, 6]
[6, 7, 8] [7, 8, 9]
[9, 10, 11] [10, 11, 12]
[12, 13, 14] [13, 14, 15]
[15, 16, 17] [16, 17, 18]


## 隨機採樣的時間序列生成器

給定資料 data 與區段長度 T，會隨機打亂起始位置，

依序產生：
* X = 連續 T 筆資料
* Y = 對應往後平移 1 筆的 T 筆資料

形成 (X, Y) 配對。
* 若 repeat=False，則跑完一輪後停止；
* 若 repeat=True，則會持續無限循環產生資料。

In [3]:
import numpy as np
import random

def seg_data_iter_random_one(data, T, repeat=False):
    while True:  
        # 計算可作為起始點的最大範圍
        end = len(data) - T
        
        # 建立所有可能的起始索引
        indices = list(range(0, end))
        
        # 將索引順序隨機打亂
        random.shuffle(indices)
        
        # 依照打亂後的索引逐一產生資料
        for i in range(end):
            p = indices[i]  # 取得隨機起始位置
            
            # 取出長度為 T 的連續資料作為 X
            X = data[p:p+T]
            
            # 取出往後平移 1 格的資料作為 Y
            Y = data[p+1:p+T+1] 
            
            # 使用 yield 產生生成器結果 (X, Y)
            yield X, Y 
        
        # 若不需重複產生，則結束函式
        if not repeat:
            return

範例：本段程式使用 seg_data_iter_random_one 產生隨機連續區段資料，# 每次取得長度為 3 的 X，以及對應往後平移 1 格的 Y。# 透過迴圈印出前三組 (X, Y) 結果後即停止。

In [4]:
data_it = seg_data_iter_random_one(data, 3)

# 設定計數器
i = 0

# 逐筆取出隨機產生的 (X, Y)
for X, Y in data_it:
    
    # 印出目前取得的資料區段
    print(X, Y)
    
    # 次數加 1
    i += 1
    
    # 只取前三組資料就跳出迴圈
    if i == 3:
        break

[13, 14, 15] [14, 15, 16]
[14, 15, 16] [15, 16, 17]
[8, 9, 10] [9, 10, 11]


## 隨機產生批次（batch）連續區段資料

給定：
* data        → 原始資料（通常為時間序列）
* T           → 每段資料長度
* batch_size  → 每一批次要產生幾組樣本
* repeat      → 是否重複無限產生資料

每次會：
1. 將所有可能起始位置打亂
2. 依 batch_size 分批取出
3. 產生：
     X = 長度為 T 的連續資料
     Y = 對應往後平移 1 格的資料
4. 以 yield 回傳 (X, Y) 批次資料

若 repeat=False，跑完一輪後即停止；
若 repeat=True，則持續循環產生資料。

In [5]:
import numpy as np
import random

def seg_data_iter_random(data, T, batch_size, repeat=False):
    while True:  
        # 計算可用的起始位置範圍
        end = len(data) - T
        
        # 建立所有可能起始索引
        indices = list(range(0, end))
        
        # 將索引順序打亂（隨機抽樣）
        random.shuffle(indices)
        
        # 依 batch_size 分批處理
        for i in range(0, end, batch_size):
            
            # 取出當前批次的索引
            batch_indices = indices[i:(i + batch_size)]
            
            # 建立 X 批次資料（每筆長度為 T）
            X = [data[p:p+T] for p in batch_indices]
            
            # 建立 Y 批次資料（往後平移 1 格）
            Y = [data[p+1:p+T+1] for p in batch_indices]
            
            # 回傳一個 batch 的 (X, Y)
            yield X, Y 
        
        # 若不需重複產生，則結束
        if not repeat:
            return

範例：本段程式使用 seg_data_iter_random 建立隨機批次資料產生器，每段資料長度為 3，每個 batch 含 2 組樣本。透過迴圈依序取出批次資料，印出前三個 batch 後即停止。

建立隨機批次資料產生器：
* 參數 3 代表每段長度 T=3
* 參數 2 代表每批次 batch_size=2

In [6]:
data_it = seg_data_iter_random(data, 3, 2)

# 設定批次計數器
i = 0

# 逐批次取出資料
for X, Y in data_it:
    
    # 印出該批次的輸入資料
    print("X:", X)
    
    # 印出該批次對應的目標資料
    print("Y:", Y)
    
    # 批次數加 1
    i += 1
    
    # 只取前三個 batch 後跳出
    if i == 3:
        break

X: [[16, 17, 18], [9, 10, 11]]
Y: [[17, 18, 19], [10, 11, 12]]
X: [[12, 13, 14], [7, 8, 9]]
Y: [[13, 14, 15], [8, 9, 10]]
X: [[13, 14, 15], [1, 2, 3]]
Y: [[14, 15, 16], [2, 3, 4]]


## 順序分塊取樣 (Sequential Partitioning / Block Sampling)

本程式示範如何將一維資料轉成「固定 batch_size 的區塊排列」，# 並建立對應的 X / Y（Y 為往右平移一格的資料）。 常見於 RNN / 時間序列訓練前的資料重排處理。

重點流程：
1. 將資料轉為 numpy array
2. 依 batch_size 重新 reshape 成 2 維
3. 計算每個 batch 可使用的有效長度 block_len
4. 產生 data_x（原始序列）
5. 產生 data_y（右移一格的對應標籤）

In [7]:
import numpy as np

# 設定 batch 大小
batch_size = 2

# 將原本 data 轉為 numpy array
data = np.array(data)

# 重新排列成 (batch_size, ?)
# -1 代表自動計算欄數
data = data.reshape(batch_size, -1)
print(data)

# 重新建立 0~19 的資料
data = np.array(range(20))
print(data)

# 設定 batch 大小
batch_size = 2

# 計算每個 batch 可使用的有效長度
# (len(data)-1) 是因為後面要做右移一格
block_len = (len(data) - 1) // 2
print(block_len)

# 取出可整除 batch_size 的資料長度
data_x = data[0:block_len * batch_size]

# reshape 成 (batch_size, block_len)
data_x = data_x.reshape(batch_size, -1)
print(data_x)

# 建立對應的 Y（整體往右平移 1 格）
data_y = data[1:1 + block_len * batch_size]

# reshape 成 (batch_size, block_len)
data_y = data_y.reshape(batch_size, -1)
print(data_y)

[[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
9
[[ 0  1  2  3  4  5  6  7  8]
 [ 9 10 11 12 13 14 15 16 17]]
[[ 1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18]]


## 連續取樣（Consecutive Sampling）

本函式為 RNN「連續取樣（consecutive sampling）」的資料產生器。主要用於時間序列模型（例如 RNN / LSTM）訓練。

* 每個 epoch 隨機決定一個起始點 start（避免每次訓練都從同一位置開始）
* 將資料依 batch_size 切成固定區塊（每列是一條連續序列）
* 建立 X（原始序列）與 Y（整體右移一格的標籤）
* 在每個 batch 區塊中，再依 seq_len 切出固定長度的小序列
* 每次 yield 一批 (X, Y)，形狀為：(batch_size, seq_len)

這種方式的特色是：
* 每個 batch 內部資料是「時間上連續的」
* 適合需要保留時間依賴性的模型

In [8]:
def rnn_data_iter_consecutive(data, batch_size, seq_len, start_range=10):
    # 產生適合 RNN 訓練的連續批次資料
    # X 為輸入序列，Y 為對應的下一步目標序列
    # 透過隨機起始位置，避免每次切分結果都完全相同
    
    # 隨機選擇起始位置，讓每個 epoch 的切分起點略有變化
    start = np.random.randint(0, start_range)    
    
    # 計算每個 batch 可分配到的有效長度
    # -1 是因為 Y 需要相對 X 往右平移 1 格
    block_len = (len(data) - start - 1) // batch_size
  
    # 建立輸入序列資料
    # 先取出可剛好分配給 batch_size 的資料長度，再 reshape 成二維
    # 每一列代表一條連續子序列
    Xs = data[start:start + block_len * batch_size]   
    Xs = Xs.reshape(batch_size, -1)
    
    # 建立目標序列資料
    # 相較於 X 整體往右平移 1 格，作為下一步預測目標
    Ys = data[start + 1:start + block_len * batch_size + 1]      
    Ys = Ys.reshape(batch_size, -1)
    
    # 計算每條序列可以切出多少個長度為 seq_len 的小片段
    # 只保留可完整切分的部分
    num_batches = Xs.shape[1] // seq_len
    
    # 可完整切分的最終位置
    end_pos = num_batches * seq_len
    
    # 依序切出固定長度的小序列
    # 每次產出一個 batch_size × seq_len 的訓練批次
    for i in range(0, end_pos, seq_len):
        
        # 取出一批輸入資料
        X = Xs[:, i:(i + seq_len)]
        
        # 取出對應的一批目標資料
        Y = Ys[:, i:(i + seq_len)]
        
        # 以 generator 方式逐批回傳
        yield X, Y

範例：建立 0~19 的資料，使用 rnn_data_iter_consecutive 產生連續取樣的 RNN 訓練資料，印出每一批 (X, Y)，最後將最後一批 X 做維度轉換（調整成 RNN 常用的時間步格式）

重點：
* 原始 X 形狀為 (batch_size, seq_len)
* 經過 swapaxes 後變成 (seq_len, batch_size)
* 再 reshape 成 (seq_len, batch_size, feature_dim)
* 常用於符合 PyTorch / 深度學習模型的輸入格式

In [9]:
import numpy as np

# 建立 0~19 的資料
data = list(range(20))

# 印出前 20 筆（實際上就是全部）
print(data[:20])

# 建立連續取樣資料產生器
batch_size = 2
seq_len = 3
start_range = 1 # 故意從 0~1 隨機選起始點，讓每次產生的資料略有不同
data_it = rnn_data_iter_consecutive(np.array(data[:20]), batch_size, seq_len, start_range)

# 逐批印出資料
for X, Y in data_it:
    print("X:", X)
    print("Y:", Y)

# 將最後一批 X 做維度轉換
# 原本形狀：(batch_size, seq_len)
x1 = np.swapaxes(X, 0, 1)  # 轉成 (seq_len, batch_size)

# 再增加一個 feature 維度
x1 = x1.reshape(x1.shape[0], x1.shape[1], -1)

# 印出轉換後結果
print(x1)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
X: [[ 0  1  2]
 [ 9 10 11]]
Y: [[ 1  2  3]
 [10 11 12]]
X: [[ 3  4  5]
 [12 13 14]]
Y: [[ 4  5  6]
 [13 14 15]]
X: [[ 6  7  8]
 [15 16 17]]
Y: [[ 7  8  9]
 [16 17 18]]
[[[ 6]
  [15]]

 [[ 7]
  [16]]

 [[ 8]
  [17]]]


## 連續取樣資料產生器（consecutive sampling）加強版

每個 epoch 會隨機選一個起始點 start（避免每次切法都一樣），把一維/序列資料切成 batch_size 條「連續的序列區塊」，再把每條序列依 seq_len 切成一段一段的訓練樣本，同時產生 X 與 Y：X：原序列片段、Y：整體右移 1 格的對應目標，透過 to_3D 決定輸出格式：

- to_3D=True：輸出 (seq_len, batch_size, feature_dim)（常見給 RNN 用的 3D 形狀）
- to_3D=False：輸出 (seq_len, batch_size)（較簡化的 2D 形狀）

每個 epoch 的第一個 batch 會回傳 reset=True（True/False），通常用來提示訓練端要不要重置 hidden state

In [10]:
# ===== 程式概要說明 =====
# 本函式是 RNN 用的「連續取樣資料產生器（consecutive sampling）」加強版：
import numpy as np

def rnn_data_iter_consecutive(data, batch_size, seq_len, start_range=10, to_3D=True):
    # 每次從不同的起點開始取樣，讓每個 epoch 的樣本略有不同
    start = np.random.randint(0, start_range)
    
    # 計算每個 batch 區塊可用的長度（-1 是因為 Y 要右移一格）
    block_len = (len(data) - start - 1) // batch_size

    # 取出可整除 batch_size 的連續資料作為 X / Y
    Xs = data[start:start + block_len * batch_size]
    Ys = data[start + 1:start + block_len * batch_size + 1]

    # reshape 成 (batch_size, block_len)：每列是一條連續序列
    Xs = Xs.reshape(batch_size, -1)
    Ys = Ys.reshape(batch_size, -1)

    # 計算每條序列可切出幾段 seq_len
    reset = True
    num_batches = Xs.shape[1] // seq_len

    # 逐段切出長度為 seq_len 的樣本
    for i in range(0, num_batches * seq_len, seq_len):
        X = Xs[:, i:(i + seq_len)]
        Y = Ys[:, i:(i + seq_len)]

        # 依需求轉成 RNN 常用格式
        if to_3D:
            # 先把維度從 (batch_size, seq_len) 換成 (seq_len, batch_size)
            X = np.swapaxes(X, 0, 1)
            Y = np.swapaxes(Y, 0, 1)

            # 再補上一個 feature 維度 => (seq_len, batch_size, feature_dim)
            X = X.reshape(X.shape[0], X.shape[1], -1)
            Y = Y.reshape(Y.shape[0], Y.shape[1], -1)
        else:
            # 只做 (batch_size, seq_len) -> (seq_len, batch_size)
            X = np.swapaxes(X, 0, 1)
            Y = np.swapaxes(Y, 0, 1)

        # 第一段資料回傳 reset=True，後續為 False
        if reset:
            reset = False
            yield X, Y, True
        else:
            yield X, Y, False

In [11]:
data = np.array(list(range(20))).reshape(-1,1)
data_it = rnn_data_iter_consecutive(data,2,3,2)
i = 0
for X,Y,_ in data_it:
    print("X:",X)
    print("Y:",Y) 
    i+=1
    if i==2 :break

X: [[[ 1]
  [10]]

 [[ 2]
  [11]]

 [[ 3]
  [12]]]
Y: [[[ 2]
  [11]]

 [[ 3]
  [12]]

 [[ 4]
  [13]]]
X: [[[ 4]
  [13]]

 [[ 5]
  [14]]

 [[ 6]
  [15]]]
Y: [[[ 5]
  [14]]

 [[ 6]
  [15]]

 [[ 7]
  [16]]]
